# 01 — The Patent Atlas

**Network characterization of the USPTO patent citation graph over time.**

This notebook establishes the baseline: how does the patent citation network grow,
what does its degree distribution look like, and how much are patents citing across
CPC section boundaries? All of this sets the stage before topology enters in Notebook 02.

---

In [1]:
# %% Imports and setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from src.utils import DATA_DIR, get_logger
from src.graph import build_citation_graph, temporal_snapshots, cpc_cross_class_edges
from src.metrics import graph_summary, degree_distribution, cpc_mixing_rate, shannon_entropy_cpc, cpc_section_flow_matrix
from src.plotting import set_style, save_figure, PALETTE, PALETTE_LIST, CPC_COLORS, CPC_LABELS, year_axis, log_log_axes

set_style()
logger = get_logger('nb01')

In [2]:
# %% Load cleaned data
patents = pd.read_parquet(DATA_DIR / 'patents.parquet')
citations = pd.read_parquet(DATA_DIR / 'citations.parquet')
cpc_map = pd.read_parquet(DATA_DIR / 'cpc_map.parquet')

print(f'Patents:   {len(patents):,}')
print(f'Citations: {len(citations):,}')
print(f'CPC map:   {len(cpc_map):,} rows')
print(f'Date range: {patents["date"].min()} to {patents["date"].max()}')

Patents:   8,451,545
Citations: 118,011,718
CPC map:   17,668,819 rows
Date range: 1976-01-06 00:00:00 to 2025-09-30 00:00:00


## 1. Network Growth Over Time

How has the patent citation network expanded year by year?

In [3]:
# %% Annual patent and citation counts
patents['year'] = pd.to_datetime(patents['date']).dt.year
citations['citing_year'] = pd.to_datetime(citations['citing_date']).dt.year

annual_patents = patents.groupby('year').size().reset_index(name='n_patents')
annual_citations = citations.groupby('citing_year').size().reset_index(name='n_citations')
annual_citations = annual_citations.rename(columns={'citing_year': 'year'})

annual = annual_patents.merge(annual_citations, on='year', how='outer').fillna(0)
annual = annual[(annual['year'] >= 1976) & (annual['year'] <= 2023)]

# Cumulative
annual['cum_patents'] = annual['n_patents'].cumsum()
annual['cum_citations'] = annual['n_citations'].cumsum()

In [4]:
# %% Figure 1: Network growth
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Annual counts
ax = axes[0]
ax.bar(annual['year'], annual['n_patents'], color=PALETTE['blue'], alpha=0.7, label='Patents granted')
ax2 = ax.twinx()
ax2.plot(annual['year'], annual['n_citations'], color=PALETTE['orange'], linewidth=2, label='Citations made')
ax.set_xlabel('Year')
ax.set_ylabel('Patents Granted', color=PALETTE['blue'])
ax2.set_ylabel('Citations Made', color=PALETTE['orange'])
ax.set_title('Annual Patent Activity')
year_axis(ax, 1976, 2023)

# Cumulative
ax = axes[1]
ax.plot(annual['year'], annual['cum_patents'] / 1e6, color=PALETTE['blue'], linewidth=2, label='Cumulative patents (M)')
ax2 = ax.twinx()
ax2.plot(annual['year'], annual['cum_citations'] / 1e6, color=PALETTE['orange'], linewidth=2, label='Cumulative citations (M)')
ax.set_xlabel('Year')
ax.set_ylabel('Patents (millions)', color=PALETTE['blue'])
ax2.set_ylabel('Citations (millions)', color=PALETTE['orange'])
ax.set_title('Cumulative Network Growth')
year_axis(ax, 1976, 2023)

fig.suptitle('Figure 1: Growth of the USPTO Patent Citation Network', fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, '01_network_growth')

2026-03-16 05:10:11 | src.plotting             | INFO    | Saved figure: /Users/christopherortiz/Desktop/the-shape-of-discovery/figures/01_network_growth.png


PosixPath('/Users/christopherortiz/Desktop/the-shape-of-discovery/figures/01_network_growth.png')

## 2. Degree Distribution Evolution

Patent citation networks are known to be scale-free. Does the degree distribution
change shape over time?

In [5]:
# %% Compute degree distributions for select years
snapshot_years = [1985, 1995, 2005, 2015, 2023]
colors = [PALETTE['blue'], PALETTE['cyan'], PALETTE['green'], PALETTE['orange'], PALETTE['red']]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for yr, color in zip(snapshot_years, colors):
    # Build cumulative graph up to that year
    mask = citations['citing_year'] <= yr
    yr_cites = citations[mask]
    graph = build_citation_graph(yr_cites)
    degs = degree_distribution(graph)
    
    # In-degree (how often cited)
    in_deg = degs['in_degree']
    in_deg = in_deg[in_deg > 0]
    vals, counts = np.unique(in_deg, return_counts=True)
    axes[0].loglog(vals, counts / counts.sum(), '.', color=color, alpha=0.5, markersize=3, label=str(yr))
    
    # Out-degree (how many references)
    out_deg = degs['out_degree']
    out_deg = out_deg[out_deg > 0]
    vals, counts = np.unique(out_deg, return_counts=True)
    axes[1].loglog(vals, counts / counts.sum(), '.', color=color, alpha=0.5, markersize=3, label=str(yr))

axes[0].set_title('In-Degree Distribution (Citations Received)')
axes[0].set_xlabel('In-degree')
axes[0].set_ylabel('Probability')
axes[0].legend()

axes[1].set_title('Out-Degree Distribution (References Made)')
axes[1].set_xlabel('Out-degree')
axes[1].set_ylabel('Probability')
axes[1].legend()

fig.suptitle('Figure 2: Degree Distribution Evolution (Log-Log)', fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, '01_degree_distributions')

2026-03-16 05:10:13 | src.graph                | INFO    | Built graph: 517636 nodes, 952709 edges


2026-03-16 05:10:13 | memory                   | INFO    | After graph construction: RSS = 7992 MB


2026-03-16 05:10:13 | timer                    | INFO    | build_citation_graph completed in 2.0s


2026-03-16 05:10:25 | src.graph                | INFO    | Built graph: 1476084 nodes, 5663734 edges


2026-03-16 05:10:25 | memory                   | INFO    | After graph construction: RSS = 7992 MB


...

2026-03-16 05:14:57 | src.graph                | INFO    | Built graph: 7490316 nodes, 108067004 edges


2026-03-16 05:14:57 | memory                   | INFO    | After graph construction: RSS = 9822 MB


2026-03-16 05:14:57 | timer                    | INFO    | build_citation_graph completed in 2m 5.7s


2026-03-16 05:15:02 | src.plotting             | INFO    | Saved figure: /Users/christopherortiz/Desktop/the-shape-of-discovery/figures/01_degree_distributions.png


PosixPath('/Users/christopherortiz/Desktop/the-shape-of-discovery/figures/01_degree_distributions.png')

## 3. CPC Mixing Rate Over Time

Are patents citing across CPC section boundaries more frequently over time?
A rising mixing rate would suggest increasing cross-disciplinary knowledge flow.

In [6]:
# %% Compute CPC mixing rate
mixing = cpc_mixing_rate(citations, cpc_map)
mixing = mixing[(mixing['year'] >= 1980) & (mixing['year'] <= 2023)]

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(mixing['year'], mixing['mixing_rate'], color=PALETTE['blue'], linewidth=2.5)
ax.fill_between(mixing['year'], 0, mixing['mixing_rate'], alpha=0.1, color=PALETTE['blue'])
ax.set_xlabel('Year')
ax.set_ylabel('Cross-Section Citation Rate')
ax.set_title('Figure 3: CPC Mixing Rate — Fraction of Citations Crossing Section Boundaries')
year_axis(ax)
ax.set_ylim(0, None)
fig.tight_layout()
save_figure(fig, '01_cpc_mixing_rate')

2026-03-16 05:17:52 | src.plotting             | INFO    | Saved figure: /Users/christopherortiz/Desktop/the-shape-of-discovery/figures/01_cpc_mixing_rate.png


PosixPath('/Users/christopherortiz/Desktop/the-shape-of-discovery/figures/01_cpc_mixing_rate.png')

## 4. CPC Section Citation Flow Heatmap

Which CPC sections cite each other most? This reveals the structure of
interdisciplinary knowledge flow.

In [7]:
# %% Section flow heatmap for three time periods
periods = [
    ('1980-1995', (1980, 1995)),
    ('1996-2010', (1996, 2010)),
    ('2011-2023', (2011, 2023)),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (label, (y1, y2)) in zip(axes, periods):
    mask = (citations['citing_year'] >= y1) & (citations['citing_year'] <= y2)
    period_cites = citations[mask]
    flow = cpc_section_flow_matrix(period_cites, cpc_map)
    
    # Normalize rows to show fraction
    flow_norm = flow.div(flow.sum(axis=1), axis=0)
    
    # Custom labels
    tick_labels = [f'{s}' for s in flow_norm.index]
    
    sns.heatmap(
        flow_norm, ax=ax, cmap='YlOrRd', annot=True, fmt='.2f',
        xticklabels=tick_labels, yticklabels=tick_labels,
        vmin=0, vmax=1, cbar_kws={'shrink': 0.8}
    )
    ax.set_title(label)
    ax.set_xlabel('Cited Section')
    ax.set_ylabel('Citing Section')

fig.suptitle('Figure 4: Citation Flow Between CPC Sections (Row-Normalized)', fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, '01_cpc_section_heatmap')

2026-03-16 05:21:15 | src.plotting             | INFO    | Saved figure: /Users/christopherortiz/Desktop/the-shape-of-discovery/figures/01_cpc_section_heatmap.png


PosixPath('/Users/christopherortiz/Desktop/the-shape-of-discovery/figures/01_cpc_section_heatmap.png')

## 5. Most-Cited Patents by Decade

Which patents are the most influential in each era?

In [8]:
# %% Top-cited patents per decade
decades = [
    ('1980s', 1980, 1989),
    ('1990s', 1990, 1999),
    ('2000s', 2000, 2009),
    ('2010s', 2010, 2019),
]

for label, y1, y2 in decades:
    mask = (citations['citing_year'] >= y1) & (citations['citing_year'] <= y2)
    decade_cites = citations[mask]
    top = decade_cites['cited_id'].value_counts().head(10)
    
    # Get patent titles
    top_df = top.reset_index()
    top_df.columns = ['patent_id', 'citation_count']
    top_df = top_df.merge(patents[['patent_id', 'title', 'cpc_primary']], on='patent_id', how='left')
    
    print(f'\n=== Top 10 Most-Cited Patents: {label} ===')
    for _, row in top_df.iterrows():
        title = str(row['title'])[:60] if pd.notna(row['title']) else 'N/A'
        print(f"  {row['patent_id']:>10s}  ({row['citation_count']:,} cites)  [{row.get('cpc_primary', '?')}]  {title}")


=== Top 10 Most-Cited Patents: 1980s ===
     4105776  (141 cites)  [C]  Proline derivatives and related compounds
     4061724  (126 cites)  [C]  Crystalline silica
     3986997  (119 cites)  [C]  Pigment-free coating compositions
     4098888  (117 cites)  [C]  Thiazolylacetamido cephalosporin type compounds
     4063220  (116 cites)  [H]  Multipoint data communication system with collision detectio
     4226898  (116 cites)  [C]  Amorphous semiconductors equivalent to crystalline semicondu
     4064521  (112 cites)  [H]  Semiconductor device having a body of amorphous silicon
     4024163  (110 cites)  [C]  Insecticides
     4217374  (110 cites)  [C]  Amorphous semiconductors equivalent to crystalline semicondu
     4258264  (110 cites)  [G]  Method of and apparatus for reading out a radiation image re



=== Top 10 Most-Cited Patents: 1990s ===
     4723129  (772 cites)  [B]  Bubble jet recording method and apparatus in which a heating
     4463359  (692 cites)  [H]  Droplet generating method and apparatus thereof
     4740796  (676 cites)  [B]  Bubble jet recording method and apparatus in which a heating
     4558333  (649 cites)  [B]  Liquid jet recording head
     4345262  (644 cites)  [B]  Ink jet recording method
     4683195  (626 cites)  [C]  Process for amplifying, detecting, and/or-cloning nucleic ac
     4313124  (625 cites)  [B]  Liquid jet recording process and liquid jet recording head
     4459600  (611 cites)  [B]  Liquid jet recording device
     4683202  (602 cites)  [C]  Process for amplifying nucleic acid sequences
     4733665  (353 cites)  [A]  Expandable intraluminal graft, and method and apparatus for 



=== Top 10 Most-Cited Patents: 2000s ===
     4683202  (1,384 cites)  [C]  Process for amplifying nucleic acid sequences
     4683195  (1,186 cites)  [C]  Process for amplifying, detecting, and/or-cloning nucleic ac
     4723129  (1,167 cites)  [B]  Bubble jet recording method and apparatus in which a heating
     4463359  (968 cites)  [H]  Droplet generating method and apparatus thereof
     4740796  (968 cites)  [B]  Bubble jet recording method and apparatus in which a heating
     4733665  (926 cites)  [A]  Expandable intraluminal graft, and method and apparatus for 
     5523520  (918 cites)  [A]  Mutant dwarfism gene of petunia
     4558333  (913 cites)  [B]  Liquid jet recording head
     4345262  (887 cites)  [B]  Ink jet recording method
     4313124  (879 cites)  [B]  Liquid jet recording process and liquid jet recording head



=== Top 10 Most-Cited Patents: 2010s ===
     7674650  (3,367 cites)  [H]  Semiconductor device and manufacturing method thereof
     7061014  (3,275 cites)  [H]  Natural-superlattice homologous single crystal thin film, me
     7297977  (3,255 cites)  [H]  Semiconductor device
     6294274  (3,246 cites)  [H]  Oxide thin film
     6727522  (3,229 cites)  [H]  Transistor and semiconductor device
     5731856  (3,224 cites)  [G]  Methods for forming liquid crystal displays including thin f
     7732819  (3,221 cites)  [H]  Semiconductor device and manufacturing method thereof
     7282782  (3,212 cites)  [H]  Combined binary oxide semiconductor device
     6563174  (3,211 cites)  [H]  Thin film transistor and matrix display device
     7064346  (3,208 cites)  [H]  Transistor and semiconductor device


## Summary

Key observations from the Patent Atlas:
1. **Network growth** — both patents and citations have grown substantially
2. **Scale-free structure** — degree distributions follow heavy-tailed patterns
3. **Cross-section mixing** — trend over time reveals whether knowledge flow is becoming more interdisciplinary
4. **Section flow patterns** — which CPC sections are most tightly coupled

These baselines set the stage for Notebook 02, where we introduce persistent homology
to look for **topological** signatures that simple network metrics might miss.